# Session Management with Strands Agents

## Overview

By default, a Strands agent stores its conversation history in memory. When the process ends or the
kernel restarts, that history is gone — the agent starts every cold start from a blank slate.

Session management fixes this. A `SessionManager` hooks into the agent lifecycle and persists
conversation state automatically, so a new agent instance with the same session ID picks the
conversation back up exactly where it left off. In this tutorial, we'll start from the in-memory
failure and work through each backend the SDK offers.

| Feature | Description |
|---------|-------------|
| **Baseline (no persistence)** | Watch an agent lose its history on restart, and see why |
| **`FileSessionManager`** | Persist conversation state as JSON on the local filesystem |
| **`S3SessionManager`** | Swap to cloud storage — same agent code, different constructor |
| **Custom `SessionRepository`** | Implement your own backend, here a DynamoDB single-table design |

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* AWS account
* Amazon Bedrock model access, [guide](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access-modify.html)
* IAM permissions for Amazon S3 and Amazon DynamoDB — this notebook creates and then deletes a bucket and a table

Let's now install the required packages

In [ ]:
%pip install -r requirements.txt -q

### Importing dependency packages

Now let's import the dependency packages and resolve the AWS region and account ID

In [ ]:
import os
import boto3

REGION = boto3.session.Session().region_name or "us-east-1"
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

print(f"Region: {REGION}, Account: {ACCOUNT_ID}")

## Part 1 — Baseline: The Problem

An agent without a session manager holds all conversation state in an in-memory list.
Creating a new agent instance — which is what happens on every cold start — means
starting with an empty history.

In [ ]:
from strands import Agent

agent = Agent(
    system_prompt="You are a helpful assistant. Remember details the user shares with you."
)

# Tell the agent something memorable
response = agent("My favorite programming language is Python and I live in Seattle.")
print(response)

In [ ]:
# Within the same session the agent remembers
response = agent("What's my favorite programming language?")
print(response)

In [ ]:
# Simulate a restart: create a fresh agent instance
agent = Agent(
    system_prompt="You are a helpful assistant. Remember details the user shares with you."
)

# History is gone — the agent can't answer
response = agent("What's my favorite programming language?")
print(response)

The agent has no memory of the previous conversation because the state was never persisted.
The session managers below fix this.

## Part 2 — FileSessionManager: Local Persistence

`FileSessionManager` writes conversation state as JSON files on the local filesystem.
Pass it to the agent's `session_manager` parameter — no other code changes required.

### Filesystem layout

```
/<storage_dir>/
└── session_<session_id>/
    ├── session.json
    └── agents/
        └── agent_<agent_id>/
            ├── agent.json
            └── messages/
                ├── message_0.json
                └── message_1.json
```

In [ ]:
from strands import Agent
from strands.session.file_session_manager import FileSessionManager

SESSION_ID = "my-first-session"
STORAGE_DIR = "./sessions"

session_manager = FileSessionManager(
    session_id=SESSION_ID,
    storage_dir=STORAGE_DIR,
)

agent = Agent(
    system_prompt="You are a helpful assistant. Remember details the user shares with you.",
    session_manager=session_manager,
)

response = agent("My name is Alex and I'm building a chatbot for my startup.")
print(response)

In [ ]:
response = agent("The startup is called BrightBot and we focus on customer support automation.")
print(response)

### Inspect the persisted data

In [ ]:
import json

for root, dirs, files in os.walk(STORAGE_DIR):
    level = root.replace(STORAGE_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        print(f"{indent}  {file}")

In [ ]:
session_file = os.path.join(STORAGE_DIR, f"session_{SESSION_ID}", "session.json")
with open(session_file) as f:
    print(json.dumps(json.load(f), indent=2))

### Restore after a simulated restart

Create a new agent with the **same session ID**. `FileSessionManager` loads the
persisted conversation automatically.

In [ ]:
restored_agent = Agent(
    system_prompt="You are a helpful assistant. Remember details the user shares with you.",
    session_manager=FileSessionManager(session_id=SESSION_ID, storage_dir=STORAGE_DIR),
)

response = restored_agent("What's my name and what company do I work for?")
print(response)

In [ ]:
# Cleanup local session files before moving to the S3 section
import shutil
shutil.rmtree(STORAGE_DIR, ignore_errors=True)
print("Local session files removed.")

## Part 3 — S3SessionManager: Same Agent, Cloud Backend

`S3SessionManager` uses the same session contract as `FileSessionManager`
but stores data in S3. The only difference is the constructor — all agent
code stays identical.

### S3 key layout

```
s3://<bucket>/<prefix>/
└── session_<session_id>/
    ├── session.json
    └── agents/
        └── agent_<agent_id>/
            ├── agent.json
            └── messages/
                ├── message_0.json
                └── message_1.json
```

In [ ]:
S3_BUCKET = f"strands-sessions-tutorial-{ACCOUNT_ID}"
S3_PREFIX = "tutorial-sessions"
S3_SESSION_ID = "s3-demo-session"

s3 = boto3.client("s3", region_name=REGION)

try:
    if REGION == "us-east-1":
        s3.create_bucket(Bucket=S3_BUCKET)
    else:
        s3.create_bucket(
            Bucket=S3_BUCKET,
            CreateBucketConfiguration={"LocationConstraint": REGION},
        )
    print(f"Bucket '{S3_BUCKET}' created.")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"Bucket '{S3_BUCKET}' already exists.")

In [ ]:
from strands import Agent
from strands.session.s3_session_manager import S3SessionManager

# Constructor swap — everything else is identical to the FileSessionManager example.
# boto_session is optional; omit it to use the default credential chain.
session_manager = S3SessionManager(
    session_id=S3_SESSION_ID,
    bucket=S3_BUCKET,
    prefix=S3_PREFIX,
    boto_session=boto3.Session(region_name=REGION),
)

agent = Agent(
    system_prompt="You are a helpful assistant. Remember details the user shares with you.",
    session_manager=session_manager,
)

response = agent("I'm working on a machine learning pipeline that processes satellite imagery.")
print(response)

In [ ]:
response = agent("The pipeline uses SageMaker for training and Lambda for inference.")
print(response)

### Inspect objects in S3

In [ ]:
response_s3 = s3.list_objects_v2(
    Bucket=S3_BUCKET,
    Prefix=f"{S3_PREFIX}/session_{S3_SESSION_ID}/",
)
print("Objects in S3:")
for obj in response_s3.get("Contents", []):
    print(f"  {obj['Key']}  ({obj['Size']} bytes)")

### Restore after a simulated restart

In [ ]:
restored_agent = Agent(
    system_prompt="You are a helpful assistant. Remember details the user shares with you.",
    session_manager=S3SessionManager(
        session_id=S3_SESSION_ID,
        bucket=S3_BUCKET,
        prefix=S3_PREFIX,
        boto_session=boto3.Session(region_name=REGION),
    ),
)

response = restored_agent("What services does my pipeline use?")
print(response)

In [ ]:
# Cleanup S3 resources
response_s3 = s3.list_objects_v2(
    Bucket=S3_BUCKET,
    Prefix=f"{S3_PREFIX}/session_{S3_SESSION_ID}/",
)
for obj in response_s3.get("Contents", []):
    s3.delete_object(Bucket=S3_BUCKET, Key=obj["Key"])
s3.delete_bucket(Bucket=S3_BUCKET)
print(f"Bucket '{S3_BUCKET}' and all session objects deleted.")

## Part 4 — Custom Backend: DynamoDB (Advanced)

Both built-in managers implement the `SessionRepository` interface.
You can implement that interface yourself to use any storage system.
This section builds a DynamoDB backend using a single-table design.

### Table schema

| Attribute | Type | Description |
|-----------|------|-------------|
| `pk` | String (Partition Key) | `SESSION#<session_id>` |
| `sk` | String (Sort Key) | `META`, `AGENT#<agent_id>`, or `MSG#<agent_id>#<message_id>` |
| `data` | Map | Serialized session, agent, or message payload |

In [ ]:
TABLE_NAME = "strands-sessions-tutorial"
dynamodb = boto3.client("dynamodb", region_name=REGION)

try:
    dynamodb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[
            {"AttributeName": "pk", "KeyType": "HASH"},
            {"AttributeName": "sk", "KeyType": "RANGE"},
        ],
        AttributeDefinitions=[
            {"AttributeName": "pk", "AttributeType": "S"},
            {"AttributeName": "sk", "AttributeType": "S"},
        ],
        BillingMode="PAY_PER_REQUEST",
    )
    dynamodb.get_waiter("table_exists").wait(TableName=TABLE_NAME)
    print(f"Table '{TABLE_NAME}' created.")
except dynamodb.exceptions.ResourceInUseException:
    print(f"Table '{TABLE_NAME}' already exists.")

In [ ]:
from typing import Any
from boto3.dynamodb.conditions import Key
from strands.session.session_repository import SessionRepository
from strands.types.session import Session, SessionAgent, SessionMessage


class DynamoDBSessionRepository(SessionRepository):
    """DynamoDB-backed session repository using a single-table design."""

    def __init__(self, table_name: str):
        self.table = boto3.resource("dynamodb", region_name=REGION).Table(table_name)

    def _pk(self, session_id: str) -> str:
        return f"SESSION#{session_id}"

    def create_session(self, session: Session, **kwargs: Any) -> Session:
        self.table.put_item(
            Item={"pk": self._pk(session.session_id), "sk": "META", "data": session.to_dict()}
        )
        return session

    def read_session(self, session_id: str, **kwargs: Any) -> Session | None:
        item = self.table.get_item(
            Key={"pk": self._pk(session_id), "sk": "META"}
        ).get("Item")
        return Session.from_dict(item["data"]) if item else None

    def delete_session(self, session_id: str, **kwargs: Any) -> None:
        resp = self.table.query(
            KeyConditionExpression=Key("pk").eq(self._pk(session_id))
        )
        with self.table.batch_writer() as batch:
            for item in resp.get("Items", []):
                batch.delete_item(Key={"pk": item["pk"], "sk": item["sk"]})

    def create_agent(self, session_id: str, session_agent: SessionAgent, **kwargs: Any) -> None:
        self.table.put_item(Item={
            "pk": self._pk(session_id),
            "sk": f"AGENT#{session_agent.agent_id}",
            "data": session_agent.to_dict(),
        })

    def read_agent(self, session_id: str, agent_id: str, **kwargs: Any) -> SessionAgent | None:
        item = self.table.get_item(
            Key={"pk": self._pk(session_id), "sk": f"AGENT#{agent_id}"}
        ).get("Item")
        return SessionAgent.from_dict(item["data"]) if item else None

    def update_agent(self, session_id: str, session_agent: SessionAgent, **kwargs: Any) -> None:
        self.create_agent(session_id, session_agent)

    def create_message(
        self, session_id: str, agent_id: str, session_message: SessionMessage, **kwargs: Any
    ) -> None:
        self.table.put_item(Item={
            "pk": self._pk(session_id),
            "sk": f"MSG#{agent_id}#{int(session_message.message_id):010d}",
            "data": session_message.to_dict(),
        })

    def read_message(
        self, session_id: str, agent_id: str, message_id: int, **kwargs: Any
    ) -> SessionMessage | None:
        item = self.table.get_item(
            Key={"pk": self._pk(session_id), "sk": f"MSG#{agent_id}#{int(message_id):010d}"}
        ).get("Item")
        return SessionMessage.from_dict(item["data"]) if item else None

    def update_message(
        self, session_id: str, agent_id: str, session_message: SessionMessage, **kwargs: Any
    ) -> None:
        self.create_message(session_id, agent_id, session_message)

    def list_messages(
        self,
        session_id: str,
        agent_id: str,
        limit: int | None = None,
        offset: int = 0,
        **kwargs: Any,
    ) -> list[SessionMessage]:
        # Note: for production use, handle DynamoDB pagination for large histories
        resp = self.table.query(
            KeyConditionExpression=Key("pk").eq(self._pk(session_id))
            & Key("sk").begins_with(f"MSG#{agent_id}#"),
        )
        messages = sorted(
            [SessionMessage.from_dict(item["data"]) for item in resp.get("Items", [])],
            key=lambda m: m.message_id,
        )
        messages = messages[int(offset):]
        return messages[:int(limit)] if limit is not None else messages


print("DynamoDBSessionRepository defined.")

In [ ]:
from strands import Agent
from strands.session.repository_session_manager import RepositorySessionManager

DYNAMO_SESSION_ID = "dynamodb-demo-session"

session_manager = RepositorySessionManager(
    session_id=DYNAMO_SESSION_ID,
    session_repository=DynamoDBSessionRepository(table_name=TABLE_NAME),
)

agent = Agent(
    system_prompt="You are a helpful assistant. Remember details the user shares with you.",
    session_manager=session_manager,
)

response = agent("I'm building a real-time analytics dashboard using Kinesis and OpenSearch.")
print(response)

In [ ]:
# Restore from DynamoDB — new agent, same session ID
restored_agent = Agent(
    system_prompt="You are a helpful assistant. Remember details the user shares with you.",
    session_manager=RepositorySessionManager(
        session_id=DYNAMO_SESSION_ID,
        session_repository=DynamoDBSessionRepository(table_name=TABLE_NAME),
    ),
)

response = restored_agent("What AWS services am I using for my project?")
print(response)

In [ ]:
# Cleanup DynamoDB table
boto3.client("dynamodb", region_name=REGION).delete_table(TableName=TABLE_NAME)
print(f"Table '{TABLE_NAME}' deleted.")